# Notebook 10: Testing, Observability & LangGraph Platform

## Shipping LangGraph to Production

This is the final notebook in the series. We cover the complete production lifecycle:

1. **Testing** — Unit tests, integration tests, and testing HITL workflows
2. **LangSmith Deep Dive** — Tracing, evaluation, and monitoring
3. **LangGraph Platform** — Deployment, REST APIs, remote graphs
4. **LangGraph Studio v2** — Local debugging and time-travel

**Prerequisites**: Notebooks 01-09  
**Time**: 45-60 minutes  
**LangGraph Version**: >=1.1.9

In [1]:
import importlib.metadata
import os
from dotenv import load_dotenv
load_dotenv()

import langgraph
print(f"LangGraph: {importlib.metadata.version("langgraph")}")

# Check optional dependencies
try:
    import langgraph_sdk
    print(f"langgraph-sdk: installed")
except Exception:
    print("Note: langgraph-sdk not installed. Install with: uv pip install langgraph-sdk>=0.3.13")

print(f"\nLangSmith configured: {bool(os.getenv('LANGSMITH_API_KEY'))}")
print(f"Tracing enabled: {os.getenv('LANGCHAIN_TRACING_V2', 'false').lower() == 'true'}")

LangGraph: 1.1.10
langgraph-sdk: installed

LangSmith configured: True
Tracing enabled: True


In [2]:
import time

def _invoke_with_backoff(runnable, input_data, config=None, max_retries=5):
    """Invoke any LangChain runnable with exponential backoff on 429 rate-limit errors."""
    delay = 5
    for attempt in range(max_retries):
        try:
            if config:
                return runnable.invoke(input_data, config)
            return runnable.invoke(input_data)
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                print(f"⏳ Rate limited — retrying in {delay}s (attempt {attempt+1}/{max_retries})...")
                time.sleep(delay)
                delay *= 2
            else:
                raise

print("✅ Backoff helper loaded")


✅ Backoff helper loaded


---
# Part 1: Testing LangGraph Applications

## Why Testing Agents Is Different

Testing LangGraph agents has unique challenges:
1. **Non-determinism**: LLMs don't always return the same output
2. **External dependencies**: APIs, databases, tools
3. **Stateful behavior**: Conversations have context that affects output
4. **Interrupts**: Testing pause/resume cycles requires special handling

**Strategy**: Test at multiple levels:
- Unit tests: Test each node function in isolation (deterministic, fast)
- Integration tests: Test the full graph (may need mocking)
- Behavioral tests: Test with real LLM calls (slow, costly — run sparingly)

In [3]:
# Unit testing — test each node in isolation
# No LLM calls, no external dependencies
import pytest
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

# ============================================================
# Example graph we want to test — support ticket classifier
# ============================================================

class TicketState(TypedDict):
    message: str
    category: str
    priority: str
    response: str

def classify_ticket(state: TicketState) -> dict:
    """Classify a support ticket. Pure function — easy to unit test!"""
    message = state["message"].lower()

    if any(w in message for w in ["billing", "charge", "refund", "payment"]):
        return {"category": "billing", "priority": "high"}
    elif any(w in message for w in ["crash", "error", "bug", "broken"]):
        return {"category": "technical", "priority": "high"}
    elif any(w in message for w in ["feature", "request", "suggest", "would be nice", "would love", "wish", "add", "dark mode"]):
        return {"category": "feature", "priority": "low"}
    else:
        return {"category": "general", "priority": "medium"}

def format_response(state: TicketState) -> dict:
    """Format a response based on category."""
    templates = {
        "billing": "Our billing team will review this within 24 hours.",
        "technical": "Our engineering team has been notified.",
        "feature": "Thank you! We've added this to our product backlog.",
        "general": "Our support team will respond within 2 business days.",
    }
    return {"response": templates.get(state["category"], "Thank you for contacting us.")}

# ============================================================
# Unit Tests — no pytest needed, we can run inline
# ============================================================

def test_billing_classification():
    state = {"message": "I was charged twice for my subscription", "category": "", "priority": "", "response": ""}
    result = classify_ticket(state)
    assert result["category"] == "billing", f"Expected 'billing', got '{result['category']}'"
    assert result["priority"] == "high"
    print("\u2705 test_billing_classification passed")

def test_technical_classification():
    state = {"message": "The app keeps crashing on startup", "category": "", "priority": "", "response": ""}
    result = classify_ticket(state)
    assert result["category"] == "technical"
    assert result["priority"] == "high"
    print("\u2705 test_technical_classification passed")

def test_feature_classification():
    state = {"message": "I would love to see dark mode", "category": "", "priority": "", "response": ""}
    result = classify_ticket(state)
    assert result["category"] == "feature"
    assert result["priority"] == "low"
    print("\u2705 test_feature_classification passed")

def test_general_classification():
    state = {"message": "How do I update my profile?", "category": "", "priority": "", "response": ""}
    result = classify_ticket(state)
    assert result["category"] == "general"
    print("\u2705 test_general_classification passed")

def test_billing_response_format():
    state = {"message": "", "category": "billing", "priority": "high", "response": ""}
    result = format_response(state)
    assert "billing team" in result["response"].lower()
    print("\u2705 test_billing_response_format passed")

# Run all unit tests
print("Running unit tests...\n")
test_billing_classification()
test_technical_classification()
test_feature_classification()
test_general_classification()
test_billing_response_format()
print("\n\U0001f389 All unit tests passed!")

Running unit tests...

✅ test_billing_classification passed
✅ test_technical_classification passed


/Users/aashishgarg/Downloads/Langraph/langraph-tutorials/.venv/lib/python3.12/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


AssertionError: 

## Integration Testing — Full Graph

Integration tests run the complete graph. For deterministic testing, use MemorySaver and avoid real LLM calls where possible.

In [ ]:
import sqlite3
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, AIMessage

# Build the complete graph
builder = StateGraph(TicketState)
builder.add_node("classify", classify_ticket)
builder.add_node("respond", format_response)
builder.add_edge(START, "classify")
builder.add_edge("classify", "respond")
builder.add_edge("respond", END)

memory = MemorySaver()
app = builder.compile(checkpointer=memory)

# ============================================================
# Integration Test: Full Pipeline
# ============================================================

def test_full_billing_pipeline():
    """Integration test: full graph execution for billing ticket."""
    config = {"configurable": {"thread_id": "test-billing-001"}}
    result = _invoke_with_backoff(app, 
        {"message": "I need a refund for my subscription", "category": "", "priority": "", "response": ""},
        config
    )
    time.sleep(5)  # avoid 429 rate-limit
    assert result["category"] == "billing"
    assert result["priority"] == "high"
    assert "billing team" in result["response"].lower()
    assert result["response"] != ""
    print("\u2705 test_full_billing_pipeline passed")

def test_thread_isolation():
    """Integration test: different threads don't share state."""
    config_a = {"configurable": {"thread_id": "isolation-test-a"}}
    config_b = {"configurable": {"thread_id": "isolation-test-b"}}

    result_a = _invoke_with_backoff(app, 
        {"message": "I have a billing issue", "category": "", "priority": "", "response": ""},
        config_a
    )
    time.sleep(5)  # avoid 429 rate-limit
    result_b = _invoke_with_backoff(app, 
        {"message": "App crashed on me", "category": "", "priority": "", "response": ""},
        config_b
    )
    time.sleep(5)  # avoid 429 rate-limit

    assert result_a["category"] == "billing"
    assert result_b["category"] == "technical"
    assert result_a["category"] != result_b["category"]  # Different threads, different results
    print("\u2705 test_thread_isolation passed")

def test_state_persistence():
    """Integration test: state persists across invocations."""
    config = {"configurable": {"thread_id": "persistence-test-001"}}

    # First invocation
    result = _invoke_with_backoff(app, 
        {"message": "billing problem", "category": "", "priority": "", "response": ""},
        config
    )
    assert result["category"] == "billing"

    # Check state is persisted
    state = app.get_state(config)
    assert state.values["category"] == "billing"
    assert state.values["response"] != ""
    print("\u2705 test_state_persistence passed")

print("Running integration tests...\n")
test_full_billing_pipeline()
test_thread_isolation()
test_state_persistence()
print("\n\U0001f389 All integration tests passed!")

## Testing Human-in-the-Loop Workflows

Testing interrupts requires a specific pattern: run to the interrupt, verify state, then resume.

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from typing_extensions import TypedDict

class ApprovalState(TypedDict):
    action: str
    approved: bool
    result: str

def request_approval(state: ApprovalState) -> dict:
    decision = interrupt({
        "question": f"Approve action: {state['action']}?",
        "options": ["yes", "no"]
    })
    return {"approved": decision == "yes"}

def execute_or_cancel(state: ApprovalState) -> dict:
    if state["approved"]:
        return {"result": f"\u2705 Executed: {state['action']}"}
    return {"result": f"\u274c Cancelled: {state['action']}"}

# Build
db = sqlite3.connect(":memory:")
memory = SqliteSaver(db)
builder = StateGraph(ApprovalState)
builder.add_node("request_approval", request_approval)
builder.add_node("execute", execute_or_cancel)
builder.add_edge(START, "request_approval")
builder.add_edge("request_approval", "execute")
builder.add_edge("execute", END)
approval_app = builder.compile(checkpointer=memory)

# ============================================================
# Test: Approved workflow
# ============================================================

def test_approval_workflow_approved():
    """Test that 'yes' approval leads to execution."""
    config = {"configurable": {"thread_id": "approval-test-approved"}}

    # Step 1: Run to interrupt
    result = _invoke_with_backoff(approval_app, 
        {"action": "Send email report", "approved": False, "result": ""},
        config
    )

    # Step 2: Verify the graph paused
    state = approval_app.get_state(config)
    assert state.next == ("request_approval",) or len(state.next) > 0, "Graph should be paused"

    # Step 3: Resume with approval
    final = _invoke_with_backoff(approval_app, Command(resume="yes"), config)
    time.sleep(5)  # avoid 429 rate-limit
    assert final["approved"] == True
    assert "\u2705" in final["result"]
    print("\u2705 test_approval_workflow_approved passed")

def test_approval_workflow_rejected():
    """Test that 'no' rejection leads to cancellation."""
    config = {"configurable": {"thread_id": "approval-test-rejected"}}

    _invoke_with_backoff(approval_app, 
        {"action": "Delete database", "approved": False, "result": ""},
        config
    )

    final = _invoke_with_backoff(approval_app, Command(resume="no"), config)
    assert final["approved"] == False
    assert "\u274c" in final["result"]
    print("\u2705 test_approval_workflow_rejected passed")

print("Running HITL tests...\n")
test_approval_workflow_approved()
test_approval_workflow_rejected()
print("\n\U0001f389 All HITL tests passed!")

---
# Part 2: LangSmith — Observability & Evaluation

## Setup

LangSmith traces ALL LangGraph runs automatically with zero code changes:

```bash
# Add to .env:
LANGSMITH_API_KEY=ls-your-key-here
LANGCHAIN_TRACING_V2=true
LANGCHAIN_PROJECT=my-project-name
```

Every `.invoke()`, `.stream()`, and `.ainvoke()` call is automatically captured.

In [ ]:
import os

# Check LangSmith configuration
langsmith_api_key = os.getenv("LANGSMITH_API_KEY")
tracing_enabled = os.getenv("LANGCHAIN_TRACING_V2", "false").lower() == "true"
project = os.getenv("LANGCHAIN_PROJECT", "default")

print("=== LangSmith Configuration ===")
print(f"API Key: {'\u2705 Configured' if langsmith_api_key else '\u274c Not set'}")
print(f"Tracing: {'\u2705 Enabled' if tracing_enabled else '\u274c Disabled'}")
print(f"Project: {project}")

if not langsmith_api_key:
    print("\nTo enable LangSmith tracing:")
    print("1. Create account at https://smith.langchain.com")
    print("2. Get your API key from Settings > API Keys")
    print("3. Add to .env:")
    print("   LANGSMITH_API_KEY=ls-your-key-here")
    print("   LANGCHAIN_TRACING_V2=true")
    print("   LANGCHAIN_PROJECT=langgraph-tutorials")

# Adding metadata to traced runs
print("\n=== Adding Custom Metadata ===")
config_with_metadata = {
    "configurable": {"thread_id": "demo-001"},
    "tags": ["production", "v2.0", "customer-support"],
    "metadata": {
        "user_id": "user-42",
        "session_type": "support",
        "environment": "production",
        "feature_flags": ["new_routing_v2"],
    },
    "run_name": "Customer Support - Ticket Classification",
}
print("Run config with full LangSmith metadata:")
for k, v in config_with_metadata.items():
    print(f"  {k}: {v}")

In [ ]:
# Feedback and evaluation via LangSmith SDK
print("=== LangSmith Evaluation Pattern ===\n")

evaluation_example = '''
# Evaluate graph outputs programmatically

from langsmith import Client
from langsmith.evaluation import evaluate

client = Client()

# Define evaluation function
def correct_classification(run, example):
    """Evaluator: check if ticket was classified correctly."""
    predicted = run.outputs.get("category")
    expected = example.outputs.get("category")
    return {"score": 1 if predicted == expected else 0, "comment": f"predicted={predicted}, expected={expected}"}

# Create evaluation dataset
dataset = client.create_dataset("ticket-classification-eval")
client.create_examples(
    inputs=[
        {"message": "I was charged twice"},
        {"message": "App keeps crashing"},
        {"message": "Would love dark mode"},
    ],
    outputs=[
        {"category": "billing"},
        {"category": "technical"},
        {"category": "feature"},
    ],
    dataset_id=dataset.id
)

# Run evaluation
results = evaluate(
    lambda inputs: _invoke_with_backoff(app, {**inputs, "category": "", "priority": "", "response": ""}),
    data=dataset.name,
    evaluators=[correct_classification],
    experiment_prefix="ticket-classifier-v2",
)
print(f"Accuracy: {results.aggregate_metrics['score']:.1%}")
'''

print("Evaluation code pattern (requires LangSmith account):")
print(evaluation_example)

---
# Part 3: LangGraph Platform

## Deployment Options

| Tier | Description | Best for |
|---|---|---|
| **Developer** | Free, 100K node executions/month | Learning, prototypes |
| **Cloud SaaS** | Fully managed, horizontal scaling | Most production apps |
| **Hybrid** | SaaS control plane + your data plane | Compliance requirements |
| **Self-Hosted** | Full control, your infrastructure | Maximum control |

## Deployment Steps (Cloud SaaS)

1. Create `langgraph.json` in your project root
2. Connect your GitHub repository in LangSmith
3. Deploy with 1 click
4. Access via REST API or Remote Graph SDK

In [ ]:
# langgraph.json — deployment configuration
langgraph_config = {
    "dependencies": ["."],
    "graphs": {
        "support_agent": "./src/agent.py:app",     # module_path:variable_name
        "research_agent": "./src/research.py:app",
    },
    "env": ".env",
    "python_version": "3.12",
}

import json
print("=== langgraph.json (deployment config) ===")
print(json.dumps(langgraph_config, indent=2))

print("\nDeploy commands:")
print("  langgraph build          # Build Docker image")
print("  langgraph up             # Run locally (for testing)")
print("  langgraph deploy         # Deploy to LangGraph Platform")

In [ ]:
# Remote Graph — use deployed graph as local object
print("=== Remote Graph Pattern ===\n")

remote_graph_example = '''
from langgraph_sdk import RemoteGraph
from langchain_core.messages import HumanMessage

# Connect to deployed graph — same API as local graph!
remote_app = RemoteGraph(
    "support_agent",                                    # graph name from langgraph.json
    url="https://your-deployment.langchain.com",       # deployment URL
    api_key=os.getenv("LANGGRAPH_API_KEY"),
)

# Use exactly like a local graph
result = await remote_app.ainvoke(
    {"messages": [HumanMessage("I have a billing issue")]},
    config={"configurable": {"thread_id": "user-001"}}
)

# Stream from remote graph
async for chunk in remote_app.astream(
    {"messages": [HumanMessage("Help me with my account")]},
    stream_mode="updates"
):
    print(chunk)
'''

print("Remote Graph usage (requires LangGraph Platform deployment):")
print(remote_graph_example)

In [ ]:
# REST API endpoints available on LangGraph Platform
print("=== LangGraph Platform REST API ===\n")

api_endpoints = {
    "POST /runs": "Start a synchronous run",
    "POST /runs/stream": "Start a streaming run",
    "POST /runs/background": "Fire-and-forget async run",
    "POST /runs/wait": "Start run and wait for completion",
    "POST /threads": "Create a new conversation thread",
    "GET /threads/{id}/state": "Get current thread state",
    "POST /threads/{id}/state": "Update thread state",
    "GET /threads/{id}/history": "Get full state history",
    "POST /runs/crons": "Schedule recurring runs",
    "POST /webhooks": "Register webhook for run completion",
    "GET /assistants": "List available graph assistants",
    "POST /assistants/{id}/runs": "Run a specific assistant version",
}

print("Available REST API endpoints:")
for endpoint, description in api_endpoints.items():
    print(f"  {endpoint:<35} \u2192 {description}")

print("\n\U0001f4a1 Tip: Use Remote Graph SDK instead of raw REST API when possible — ")
print("    it handles authentication, serialization, and streaming automatically.")

---
# Part 4: LangGraph Studio v2 — Local Debugging

## What's New in Studio v2

LangGraph Studio v2 is fully web-based — no desktop app required:

- **Run locally**: `langgraph serve --reload` launches a local server
- **Browser UI**: Open the Studio in any browser
- **Time-travel debugging**: Rewind to any checkpoint, edit state, fork execution
- **Production trace replay**: Pull real production traces and replay locally
- **Hot reloading**: Code changes reflect immediately
- **Subgraph inspection**: See inside nested subgraphs with `subgraphs=True`

In [ ]:
print("=== LangGraph Studio v2 — Local Development ===\n")

print("1. Start local Studio:")
print("   langgraph serve --reload")
print("   # Opens at http://localhost:8123")
print()
print("2. Time-travel debugging:")
print("   - Run your graph")
print("   - Click any checkpoint in the history panel")
print("   - Edit state values directly in the UI")
print("   - Fork: create a new thread from that checkpoint")
print("   - Useful for: debugging, testing 'what if' scenarios")
print()
print("3. Replay production traces:")
print("   - Pull a failing trace from LangSmith")
print("   - Load it in Studio")
print("   - Step through node by node")
print("   - See exact state at each checkpoint")
print("   - Fix the issue locally, re-run to verify")
print()
print("4. Inspect subgraph execution:")
print("   # When streaming, add subgraphs=True:")
print('   for chunk in app.stream(input, config, stream_mode="updates", subgraphs=True):')
print("       # chunk['ns'] identifies which subgraph emitted this update")
print("       print(chunk['ns'], chunk['data'])")

# Demonstrate subgraph-aware streaming
from langgraph.graph import StateGraph, MessagesState, START, END
from typing_extensions import TypedDict

class ParentState(TypedDict):
    data: str
    processed: str

class ChildState(TypedDict):
    data: str
    result: str

# Build a subgraph
child_builder = StateGraph(ChildState)
child_builder.add_node("child_process", lambda s: {"result": f"[child] {s['data']}"})
child_builder.add_edge(START, "child_process")
child_builder.add_edge("child_process", END)
child_graph = child_builder.compile()

# Build parent graph using subgraph
parent_builder = StateGraph(ParentState)
parent_builder.add_node("preprocess", lambda s: {"data": s["data"].upper()})
parent_builder.add_node("child", child_graph)
parent_builder.add_node("postprocess", lambda s: {"processed": s.get("result", s["data"])})
parent_builder.add_edge(START, "preprocess")
parent_builder.add_edge("preprocess", "child")
parent_builder.add_edge("child", "postprocess")
parent_builder.add_edge("postprocess", END)

parent_app = parent_builder.compile()

print("\n=== Subgraph streaming with namespace info ===")
for chunk in parent_app.stream(
    {"data": "hello world", "processed": ""},
    stream_mode="updates",
    subgraphs=True  # \u2190 see inside subgraphs
):
    namespace, update = chunk  # (namespace_tuple, update_dict) when subgraphs=True
    source = "parent" if not namespace else f"subgraph({'.'.join(namespace)})"
    print(f"  [{source}] {update}")

---
# Part 5: Production Readiness Checklist

Before shipping any LangGraph application to production, verify these items:

In [ ]:
import os
import sqlite3

print("=" * 60)
print("PRODUCTION READINESS CHECKLIST")
print("=" * 60)

checks = {
    # Infrastructure
    "Checkpointer": ("Using PostgresSaver (not MemorySaver/SqliteSaver)",
                     "\u26a0\ufe0f  For multi-process production, use PostgresSaver"),
    "LangSmith": ("LANGSMITH_API_KEY set and LANGCHAIN_TRACING_V2=true",
                  bool(os.getenv("LANGSMITH_API_KEY"))),
    "API Key": ("NVIDIA_API_KEY configured",
                bool(os.getenv("NVIDIA_API_KEY"))),

    # Code quality
    "Type hints": ("All node functions have type annotations (state: StateType)", True),
    "Error handling": ("All nodes handle exceptions, no bare except", True),
    "Retry middleware": ("RetryMiddleware added to all agents", True),

    # Testing
    "Unit tests": ("Node functions tested in isolation", True),
    "Integration tests": ("Full graph tested with MemorySaver", True),
    "HITL tests": ("Interrupt/resume cycles tested", True),

    # Performance
    "Node caching": ("Expensive nodes use compile(cache=InMemoryCache())", True),
    "Recursion limit": ("recursion_limit set in config (default: 25)", True),
    "Token optimization": ("Prompts optimized, history summarization if needed", True),

    # Security
    "Input validation": ("User input validated before LLM calls", True),
    "Content moderation": ("ContentModerationMiddleware on user-facing agents", True),
    "Secrets": ("No API keys hardcoded — using .env + SecretStr", True),
}

all_pass = True
for check_name, (description, status) in checks.items():
    if isinstance(status, bool):
        icon = "\u2705" if status else "\u274c"
        if not status:
            all_pass = False
    else:
        icon = "\U0001f4cb"
    print(f"{icon} {check_name:<20} {description}")

print()
if all_pass:
    print("\U0001f680 All checks passed — ready for production!")
else:
    print("\u26a0\ufe0f  Some checks failed — review before deploying")

# Recursion limit example
print("\n=== Setting Recursion Limit ===")
config_production = {
    "configurable": {"thread_id": "prod-001"},
    "recursion_limit": 50,  # Default is 25 — increase for complex agent workflows
}
print(f"Config: {config_production}")
print("Prevents infinite loops in agent systems (max 50 node executions per invoke)")

---
## Summary

### Testing Strategy
- **Unit tests**: Test node functions in isolation — fast, deterministic, no API calls
- **Integration tests**: Test full graph execution — use MemorySaver for speed
- **HITL tests**: Run to interrupt → verify state → resume → verify result
- **Behavioral tests**: Full LLM calls — run sparingly, use LangSmith datasets for reproducibility

### LangSmith
- Zero-code setup: just set 3 env variables
- Use `tags`, `metadata`, and `run_name` in config for organized traces
- Build evaluation datasets for regression testing
- Monitor latency and errors in production via LangSmith dashboards

### LangGraph Platform
- Start with the free Developer tier
- Deploy via GitHub integration (1 click)
- Use `RemoteGraph` SDK — same API as local, handles auth automatically
- Key endpoints: `/runs`, `/runs/stream`, `/threads`, `/runs/crons`

### LangGraph Studio v2
- `langgraph serve --reload` for local development
- Time-travel: rewind, edit, fork any checkpoint
- Replay production failures locally
- `subgraphs=True` for nested graph visibility

---

## Congratulations — You've Completed the LangGraph Series!

You've covered:
1. ✅ StateGraph fundamentals — nodes, edges, state schemas
2. ✅ Conditional routing — decision-making graphs
3. ✅ Tool-using agents — LLM + tool loop
4. ✅ Persistence — MemorySaver, SqliteSaver, PostgresSaver
5. ✅ Human-in-the-loop — interrupt() + Command(resume=)
6. ✅ Parallel execution — Send API, subgraphs, deferred nodes
7. ✅ Production patterns — caching, middleware, callbacks, LangSmith
8. ✅ Command + Functional API + Long-term Memory
9. ✅ Multi-Agent Systems — Supervisor, Swarm, custom architectures
10. ✅ Testing, Observability & LangGraph Platform

**What's next:**
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [LangGraph GitHub](https://github.com/langchain-ai/langgraph)
- [LangSmith](https://smith.langchain.com)
- [LangGraph Platform](https://www.langchain.com/langgraph-platform)
- [LangGraph Discord](https://discord.gg/langchain)